# 11. Cloud data and discovery

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/11-cloud-and-catalogs.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=11-cloud-and-catalogs.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/11-cloud-and-catalogs.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `5359cfd27fd7baa71b8668a7f09a5680bd649fc10c660fcef4f4ba07e0d728e0`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 11: Cloud data and discovery; abstract contour illustration](assets/11-cloud-and-catalogs.svg)

# 11. Cloud data and discovery

STAC, range reads, and honest budgets

**Learning goals:** Build a local STAC Item; inspect metadata; estimate window size; distinguish browser download from remote range I/O.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=11-cloud-and-catalogs.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## A catalog points to data

STAC describes spatial-temporal assets. It does not guarantee an asset's accuracy, license, CORS configuration, or continued availability. We create a local example that points to our synthetic COG and explicitly labels its made-up date.

In [ ]:
from datetime import datetime, timezone
path = OUTPUT / "catalog-dem.tif"
raster().write(str(path), driver="COG")
left, bottom, right, top = raster().bounds
lon, lat = terrio.transform_points([left, right, right, left, left], [bottom, bottom, top, top, bottom], CRS, "EPSG:4326")
coords = [[float(x), float(y)] for x, y in zip(lon, lat)]
item = {"type": "Feature", "stac_version": "1.0.0", "id": "synthetic-terrain",
        "bbox": [min(lon), min(lat), max(lon), max(lat)], "geometry": {"type": "Polygon", "coordinates": [coords]},
        "properties": {"datetime": "2026-01-01T00:00:00Z", "title": "Synthetic terrain; illustrative timestamp"},
        "links": [], "assets": {"elevation": {"href": "catalog-dem.tif", "type": "image/tiff; application=geotiff; profile=cloud-optimized",
                                              "roles": ["data"], "title": "Original synthetic teaching data"}}}
(OUTPUT / "item.json").write_text(json.dumps(item, indent=2))
print(json.dumps(item["assets"], indent=2))

## Metadata checks before downloads

These assertions are useful local checks, not full STAC schema validation. A production workflow should validate the schema and follow the collection's license, provider information, and asset-specific terms.

In [ ]:
assert item["type"] == "Feature" and len(item["bbox"]) == 4
assert item["geometry"]["coordinates"][0][0] == item["geometry"]["coordinates"][0][-1]
assert (OUTPUT / item["assets"]["elevation"]["href"]).exists()
datetime.fromisoformat(item["properties"]["datetime"].replace("Z", "+00:00"))
print("Basic checks passed; not a full schema validation")

## Estimate before requesting

A window's pixel payload can be much smaller than an image. Actual transfers include headers and complete compressed blocks, and may involve overviews, caches, or retries. The calculated ratio is not a network benchmark.

In [ ]:
full_side, window_side, bands, bytes_per = 10980, 512, 2, 2
full = full_side**2 * bands * bytes_per
window = window_side**2 * bands * bytes_per
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["Whole image", "512 px window"], np.array([full, window])/1e6)
ax.set(ylabel="Logical uncompressed MB", title="Two uint16 bands: a planning estimate")
plt.show()
print("Payload ratio:", round(full/window, 1))

## Read the local window

TerrIO's Python browser build reads local files after download. It does not expose the native HTTP range reader. Here we test a local COG window. To test network efficiency, use the optional native recipe in the learning resources and measure bytes actually transferred.

In [ ]:
cog = terrio.open_raster(str(path), lazy=True)
tile = cog.read(1, window=(16, 16, 32, 32))
assert tile.shape == (32, 32)
image_grid([tile], ["Local COG window; no network traffic measured"])
plt.show()

## Your investigation

Find an asset in a public STAC catalog and record its provider, license, media type, acquisition date, CRS, scale/offset, and approximate size before downloading anything.

### Discussion guide

Keep catalog metadata separate from the asset's terms. An HTTP 206 response supports a range-read claim; a small in-memory array alone does not.

## Sources and credit

- [Source 1](https://stacspec.org/en)
- [Source 2](https://terrio.dev/remote/)
- [Source 3](https://terrio.dev/notebooks/)
- [Source 4](https://earth-search.aws.element84.com/v1)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.